<a href="https://colab.research.google.com/github/glebas2310/Pioneer_2_Mini/blob/main/pt_to_rknn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ПЕРЕД ВЫПОЛНЕНИЕМ ПРОГРАММЫ: СРЕДА ВЫПОЛНЕНИЯ -> СМЕНИТЬ СРЕДУ ВЫПОЛНЕНИЯ -> ГРАФИЧЕСКИЙ ПРОЦЕССОР Т4

1. Подготовка и установка библиотек

In [1]:
!pip install ultralytics roboflow -q
import os
print("✅ Базовые библиотеки установлены!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 124.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 9.3 MB/s eta 0:00:00
✅ Базовые библиотеки установлены!


2. Загрузга датасета.

---

---




RoboFlow: Versions -> Download dataset -> YOLOv8 -> Show download code -> Jupyter -> Copy


---


---
Вставьте код между горизонтальными линиями в коде.


In [2]:
from roboflow import Roboflow

# ==============================================================
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="sCDWkgncsMGaIbf45HCq")
project = rf.workspace("legorobot").project("znaki-3dnf6")
version = project.version(3)
dataset = version.download("yolov8")
# ==============================================================

dataset_yaml = f"{dataset.location}/data.yaml"
print(f"✅ Датасет готов! Путь: {dataset_yaml}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to znaki-3 in yolov8:: 100%|██████████| 908/908 [00:00<00:00, 3963.41it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
✅ Датасет готов! Путь: /content/znaki-3/data.yaml


3. Обучение best.pt и автоматический экспорт в best.onnx

In [3]:
from ultralytics import YOLO

print("🚀 Начинаем обучение YOLO11n...")
model = YOLO('yolo11n.pt')

results = model.train(data=dataset_yaml, epochs=50, imgsz=640, batch=16, plots=True)

best_pt_path = str(results.save_dir / 'weights' / 'best.pt')
print(f"✅ Модель обучена! Веса сохранены в: {best_pt_path}")

print("⚙️ Экспорт в ONNX для NPU...")
best_model = YOLO(best_pt_path)
onnx_path = best_model.export(format='onnx', opset=12, simplify=True)
print(f"✅ ONNX файл готов: {onnx_path}")

🚀 Начинаем обучение YOLO11n...
Ultralytics 8.4.157 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/znaki-3/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=

4. Установка RKNN-Toolkit2

In [8]:
%%bash
# ЯЧЕЙКА 4: Создание среды для компилятора NPU
echo "Установка менеджера uv..."
pip install uv -q

echo "Очистка старых сред и создание новой..."
rm -rf env310
uv python install 3.10
uv venv env310 --python 3.10 --seed

echo "Установка rknn-toolkit2 и зависимостей..."
# Строго фиксируем setuptools==69.5.1, чтобы вернуть модуль pkg_resources
./env310/bin/pip install "numpy<2.0.0" setuptools==69.5.1 onnx==1.16.1 rknn-toolkit2 -q

echo "✅ Среда для RKNN успешно создана!"

Установка менеджера uv...
Очистка старых сред и создание новой...
Установка rknn-toolkit2 и зависимостей...
✅ Среда для RKNN успешно создана!


Python 3.10 is already installed
Using CPython 3.10.21
Creating virtual environment with seed packages at: env310
 + packaging==26.3
 + pip==26.2.1
 + setuptools==84.0.0
 + wheel==0.48.0
Activate with: source env310/bin/activate


5. Конвертация из .onnx в .rknn

In [2]:
# ЯЧЕЙКА 5: Сборка RKNN-модели под RK3576
output_rknn = "/content/best_model.rknn"

convert_script = f"""
from rknn.api import RKNN

rknn = RKNN(verbose=False)
# Указываем реальный чип дрона — rk3576
rknn.config(mean_values=[[0, 0, 0]], std_values=[[255, 255, 255]], target_platform="rk3576")

print("Загрузка ONNX: {onnx_path}")
ret = rknn.load_onnx(model="{onnx_path}")
if ret != 0: exit(ret)

print("Сборка графа под RK3576...")
ret = rknn.build(do_quantization=False)
if ret != 0: exit(ret)

print("Сохранение модели...")
ret = rknn.export_rknn("{output_rknn}")
if ret != 0: exit(ret)

rknn.release()
"""

with open("convert.py", "w") as f:
    f.write(convert_script)

!./env310/bin/python convert.py

NameError: name 'onnx_path' is not defined